In [ ]:
"""
stage2_hawkes.py -- Stage 2 (+3): discrete-time logistic Hawkes on the JMA-pivot process.
Subcommands: fit | predict | pcont

RULES
  S2.1  p_t = P(is_target[t] | info <= t-1). Event-count features use event_bar
        in lag bins with min lag 1; age/tod computed from info <= t-1 (S2.3).
  S2.2  Lag bins (lo, hi) inclusive from --bin-edges: edges [1,2,3,5,...] ->
        bins [1,1],[2,2],[3,3],[4,5],...
  S2.3  age(t) = t - (last target bar strictly before t), session-local;
        no prior target -> age = local_t + 1. Never reads is_target[t].
  S2.4  All features are within-session => session-date splits need no embargo.
  S2.5  pcont(t, H) = prod_{h=1..H} (1 - p_{t+h}) with the event set frozen at
        <= t (counterfactual: no new oscillator or target events after t).
        Truncated at session end; h_used reported. pbreak = 1 - pcont.
  S2.6  Exogenous events with opposing == 0 dropped; TICK_HMA dropped.
  S2.7  Warm bars excluded from fit, predict and pcont query points.
  S2.8  Plain L2 logistic regression (lbfgs), no class weighting -- keeps
        probabilities on the natural base rate.
  S2.9  Optional --bootstrap N: session-level resampling with replacement,
        percentile CIs on kernel coefficients.
"""

import argparse
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression

TZ = "America/New_York"
STREAMS = ["MNQ_D1", "MNQ_D2", "TICK_D1", "TICK_D2"]
N_TOD = 14


def make_classes():
    cls = []
    for s in STREAMS:
        cls.append((s, "opp"))
        cls.append((s, "conf"))
    cls.append(("MNQ_JMA_SELF", "all"))
    return cls


def bins_from_edges(edges):
    bins, lo = [], 1
    for e in edges:
        bins.append((lo, int(e)))
        lo = int(e) + 1
    return bins


def age_bins_from_edges(edges):
    b = bins_from_edges(edges)
    b.append((int(edges[-1]) + 1, 10 ** 9))
    return b


class Featurizer:
    def __init__(self, bars, events, bin_edges, age_edges):
        self.bins = bins_from_edges(bin_edges)
        self.age_bins = age_bins_from_edges(age_edges)
        self.classes = make_classes()
        self.feature_names = (
            [f"{s}|{c}|lag{lo}_{hi}" for (s, c) in self.classes for (lo, hi) in self.bins]
            + [f"age|{lo}_{hi}" for (lo, hi) in self.age_bins]
            + [f"tod|{k}" for k in range(N_TOD)]
        )
        self.n_feat = len(self.feature_names)

        ev = events[events["stream"].isin([c[0] for c in self.classes])]
        ev = ev[~((ev["stream"] != "MNQ_JMA_SELF") & (ev["opposing"] == 0))]   # S2.6
        evg = dict(tuple(ev.groupby("session_date")))

        self.sessions = []
        bars = bars.sort_values("bar_index").reset_index(drop=True)
        for sess, g in bars.groupby("session_date", sort=True):
            n = len(g)
            first = int(g["bar_index"].iloc[0])
            tgt = g["is_target"].to_numpy()
            warm = g["warm"].to_numpy()
            ts = pd.DatetimeIndex(g["timestamp"]).tz_convert(TZ)
            mins = ts.hour.to_numpy() * 60 + ts.minute.to_numpy() - 540
            tod = np.clip(mins // 30, 0, N_TOD - 1).astype(np.int16)
            lt_incl = np.maximum.accumulate(np.where(tgt, np.arange(n), -1))
            P = {}
            e = evg.get(sess)
            for (s, c) in self.classes:
                if e is None:
                    loc = np.empty(0, np.int64)
                else:
                    if s == "MNQ_JMA_SELF":
                        sub = e[e["stream"] == s]
                    else:
                        want = 1 if c == "opp" else -1
                        sub = e[(e["stream"] == s) & (e["opposing"] == want)]
                    loc = sub["event_bar"].to_numpy() - first
                ind = np.zeros(n, np.int32)
                ind[loc] = 1
                P[(s, c)] = np.concatenate(([0], np.cumsum(ind)))              # P[i] = count pos < i
            self.sessions.append(dict(
                sess=sess, first=first, n=n, tgt=tgt, warm=warm, tod=tod,
                lt_incl=lt_incl, P=P,
                bar_index=g["bar_index"].to_numpy(),
                timestamp=g["timestamp"].to_numpy()))

    def _fill(self, S, t, out):
        n = S["n"]
        col = 0
        for c in self.classes:
            P = S["P"][c]
            for (lo, hi) in self.bins:                                         # S2.1, S2.2
                b = np.clip(t - lo + 1, 0, n)
                a = np.clip(t - hi, 0, n)
                out[:, col] = P[b] - P[a]
                col += 1
        lt = np.where(t > 0, S["lt_incl"][np.maximum(t - 1, 0)], -1)           # S2.3
        age = np.where(lt >= 0, t - lt, t + 1)
        for (lo, hi) in self.age_bins:
            out[:, col] = (age >= lo) & (age <= hi)
            col += 1
        tod = S["tod"][t]
        for k in range(N_TOD):
            out[:, col] = tod == k
            col += 1

    def _fill_frozen(self, S, q, h, out):
        """S2.5: features at t = q+h using only events/targets with pos <= q."""
        n = S["n"]
        t = q + h
        cap = q + 1
        col = 0
        for c in self.classes:
            P = S["P"][c]
            for (lo, hi) in self.bins:
                b = np.clip(np.minimum(t - lo + 1, cap), 0, n)
                a = np.clip(np.minimum(t - hi, cap), 0, n)
                out[:, col] = P[b] - P[a]
                col += 1
        lt = S["lt_incl"][q]
        age = np.where(lt >= 0, t - lt, t + 1)
        for (lo, hi) in self.age_bins:
            out[:, col] = (age >= lo) & (age <= hi)
            col += 1
        tod = S["tod"][np.minimum(t, n - 1)]
        for k in range(N_TOD):
            out[:, col] = tod == k
            col += 1

    def _selected(self, date_from, date_to):
        for S in self.sessions:
            d = str(S["sess"])
            if date_from is not None and d < date_from:
                continue
            if date_to is not None and d > date_to:
                continue
            yield S

    def build(self, date_from=None, date_to=None):
        sel = []
        total = 0
        for S in self._selected(date_from, date_to):
            t = np.nonzero(~S["warm"])[0]                                      # S2.7
            sel.append((S, t))
            total += len(t)
        X = np.zeros((total, self.n_feat), np.float32)
        y = np.zeros(total, np.int8)
        bar_index = np.zeros(total, np.int64)
        sess_id = np.zeros(total, np.int32)
        timestamp = np.zeros(total, "datetime64[ns]")
        ofs = 0
        for sid, (S, t) in enumerate(sel):
            m = len(t)
            self._fill(S, t, X[ofs:ofs + m])
            y[ofs:ofs + m] = S["tgt"][t]
            bar_index[ofs:ofs + m] = S["bar_index"][t]
            sess_id[ofs:ofs + m] = sid
            timestamp[ofs:ofs + m] = S["timestamp"][t]
            ofs += m
        meta = pd.DataFrame({"bar_index": bar_index, "timestamp": timestamp,
                             "sess_id": sess_id, "is_target": y.astype(bool)})
        return X, y, meta


def load_inputs(a):
    bars = pd.read_parquet(a.bars)
    events = pd.read_parquet(a.events)
    return bars, events


def cmd_fit(a):
    bars, events = load_inputs(a)
    edges = [int(x) for x in a.bin_edges.split(",")]
    age_edges = [int(x) for x in a.age_edges.split(",")]
    fz = Featurizer(bars, events, edges, age_edges)
    X, y, meta = fz.build(a.train_start, a.train_end)
    model = LogisticRegression(C=a.c, solver="lbfgs", max_iter=2000)           # S2.8
    model.fit(X, y)
    p = model.predict_proba(X)[:, 1]
    eps = 1e-12
    ll = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

    bundle = dict(model=model, feature_names=fz.feature_names,
                  bin_edges=edges, age_edges=age_edges, c=a.c,
                  train_start=a.train_start, train_end=a.train_end)
    joblib.dump(bundle, f"{a.out}/hawkes_model.joblib")

    coef = model.coef_[0]
    kern = pd.DataFrame({"feature": fz.feature_names, "coef": coef})
    kern["group"] = kern["feature"].str.split("|").str[0]
    kern["cls"] = kern["feature"].str.split("|").str[1]

    if a.bootstrap > 0:                                                        # S2.9
        rng = np.random.default_rng(7)
        sids = np.unique(meta["sess_id"].to_numpy())
        rows_by_sid = {s: np.nonzero(meta["sess_id"].to_numpy() == s)[0] for s in sids}
        coefs = np.zeros((a.bootstrap, fz.n_feat))
        for b in range(a.bootstrap):
            pick = rng.choice(sids, size=len(sids), replace=True)
            rows = np.concatenate([rows_by_sid[s] for s in pick])
            mb = LogisticRegression(C=a.c, solver="lbfgs", max_iter=2000)
            mb.fit(X[rows], y[rows])
            coefs[b] = mb.coef_[0]
        kern["coef_lo"] = np.percentile(coefs, 2.5, axis=0)
        kern["coef_hi"] = np.percentile(coefs, 97.5, axis=0)

    kern.to_csv(f"{a.out}/kernels.csv", index=False)
    summary = dict(n_rows=int(len(y)), n_targets=int(y.sum()),
                   base_rate=float(y.mean()), logloss_insample=float(ll),
                   intercept=float(model.intercept_[0]), n_features=fz.n_feat)
    print(json.dumps(summary, indent=2))

    if a.plot:
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots
        ks = [(s, c) for (s, c) in fz.classes]
        nc = 3
        nr = (len(ks) + nc - 1) // nc
        fig = make_subplots(rows=nr, cols=nc,
                            subplot_titles=[f"{s}|{c}" for (s, c) in ks])
        centers = [0.5 * (lo + hi) for (lo, hi) in fz.bins]
        nb = len(fz.bins)
        for i, (s, c) in enumerate(ks):
            r, cix = i // nc + 1, i % nc + 1
            w = coef[i * nb:(i + 1) * nb]
            fig.add_trace(go.Scatter(x=centers, y=w, mode="lines+markers",
                                     line_shape="hv", showlegend=False), row=r, col=cix)
            fig.add_trace(go.Scatter(x=[centers[0], centers[-1]], y=[0, 0], mode="lines",
                                     line=dict(color="black", dash="dot", width=1),
                                     showlegend=False), row=r, col=cix)
        fig.update_layout(height=320 * nr, width=1400, title="Fitted kernels (logit weight vs lag)")
        fig.write_html(f"{a.out}/kernels.html")


def cmd_predict(a):
    bundle = joblib.load(a.model)
    bars, events = load_inputs(a)
    fz = Featurizer(bars, events, bundle["bin_edges"], bundle["age_edges"])
    X, y, meta = fz.build(a.start, a.end)
    meta["p"] = bundle["model"].predict_proba(X)[:, 1]
    meta.drop(columns=["sess_id"]).to_parquet(a.out_file, index=False)
    print(json.dumps(dict(n_rows=int(len(meta)), n_targets=int(y.sum()),
                          mean_p=float(meta["p"].mean())), indent=2))


def cmd_pcont(a):
    bundle = joblib.load(a.model)
    bars, events = load_inputs(a)
    fz = Featurizer(bars, events, bundle["bin_edges"], bundle["age_edges"])
    model = bundle["model"]
    at = pd.read_parquet(a.at) if a.at.endswith(".parquet") else pd.read_csv(a.at)
    want = np.sort(at["bar_index"].to_numpy())

    out_bar, out_pc, out_hu = [], [], []
    for S in fz.sessions:
        lo, hi = S["bar_index"][0], S["bar_index"][-1]
        w = want[(want >= lo) & (want <= hi)]
        if len(w) == 0:
            continue
        q = w - S["first"]
        q = q[~S["warm"][q]]                                                   # S2.7
        if len(q) == 0:
            continue
        n = S["n"]
        logsurv = np.zeros(len(q))
        h_used = np.zeros(len(q), np.int32)
        buf = np.zeros((len(q), fz.n_feat), np.float32)
        for h in range(1, a.horizon + 1):
            valid = q + h <= n - 1
            if not valid.any():
                break
            iv = np.nonzero(valid)[0]
            self_buf = buf[:len(iv)]
            fz._fill_frozen(S, q[iv], h, self_buf)                             # S2.5
            p = model.predict_proba(self_buf)[:, 1]
            logsurv[iv] += np.log1p(-p)
            h_used[iv] += 1
        out_bar.append(q + S["first"])
        out_pc.append(np.exp(logsurv))
        out_hu.append(h_used)

    res = pd.DataFrame({"bar_index": np.concatenate(out_bar),
                        "pcont": np.concatenate(out_pc),
                        "h_used": np.concatenate(out_hu)})
    res["pbreak"] = 1.0 - res["pcont"]
    res["H"] = a.horizon
    res.to_parquet(a.out_file, index=False)
    print(json.dumps(dict(n_query=int(len(res)), mean_pbreak=float(res["pbreak"].mean()),
                          mean_h_used=float(res["h_used"].mean())), indent=2))


def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)

    f = sub.add_parser("fit")
    f.add_argument("--bars", required=True)
    f.add_argument("--events", required=True)
    f.add_argument("--out", required=True)
    f.add_argument("--train-start", default=None)
    f.add_argument("--train-end", default=None)
    f.add_argument("--bin-edges", default="1,2,3,5,8,13,21,34")
    f.add_argument("--age-edges", default="1,2,3,5,8,13,21,34,55,89")
    f.add_argument("--c", type=float, default=100.0)
    f.add_argument("--bootstrap", type=int, default=0)
    f.add_argument("--plot", action="store_true")
    f.set_defaults(fn=cmd_fit)

    p = sub.add_parser("predict")
    p.add_argument("--model", required=True)
    p.add_argument("--bars", required=True)
    p.add_argument("--events", required=True)
    p.add_argument("--start", default=None)
    p.add_argument("--end", default=None)
    p.add_argument("--out-file", required=True)
    p.set_defaults(fn=cmd_predict)

    c = sub.add_parser("pcont")
    c.add_argument("--model", required=True)
    c.add_argument("--bars", required=True)
    c.add_argument("--events", required=True)
    c.add_argument("--at", required=True)
    c.add_argument("--horizon", type=int, default=34)
    c.add_argument("--out-file", required=True)
    c.set_defaults(fn=cmd_pcont)

    a = ap.parse_args()
    a.fn(a)


if __name__ == "__main__":
    main()